# Pós-processamento de Placas — Heurísticas por Formato

Este notebook implementa e avalia heurísticas de pós-processamento para corrigir erros clássicos do OCR em placas brasileiras.

A ideia central é simples: conhecendo o **formato esperado** de cada posição da placa (letra ou número), podemos corrigir caracteres que o OCR confundiu — por exemplo, `I` onde deveria estar `1`, ou `0` onde deveria estar `O`.

O Brasil tem dois padrões vigentes:
- **Antiga**: `LLL-NNNN` (ex: `PYP-5727`) — posições 0,1,2 são letras; 3,4,5,6 são números
- **Mercosul**: `LLLNLNN` (ex: `EHJ9C82`) — posições 0,1,2 são letras; 3 é número; 4 é letra; 5,6 são números

A estratégia é puramente textual: tentamos corrigir o texto nos dois formatos e aplicamos a correção apenas quando **somente um dos formatos** produz um resultado válido. Quando ambos produzem resultados válidos (caso ambíguo), não corrigimos para evitar corrupções.

## 1. Mapeamentos e Formatos

In [ ]:
LETRA_PARA_NUMERO = {'I': '1', 'O': '0', 'S': '5', 'B': '8', 'Z': '2', 'G': '6', 'Q': '0'}
NUMERO_PARA_LETRA = {'1': 'I', '0': 'O', '5': 'S', '8': 'B', '2': 'Z'}

FORMATO_ANTIGA   = ['L', 'L', 'L', 'N', 'N', 'N', 'N']
FORMATO_MERCOSUL = ['L', 'L', 'L', 'N', 'L', 'N', 'N']

print('Mapeamentos carregados.')

## 2. Correção por Formato

A função `corrigir_por_formato` aplica substituições posição a posição com base no formato esperado.

A função `pos_processar` tenta os dois formatos e só aplica a correção quando ela resolve a ambiguidade — ou seja, quando apenas um dos formatos resulta em uma placa válida após a correção. Essa abordagem é determinística e independente de qualquer modelo treinado.

In [ ]:
def corrigir_por_formato(texto: str, fmt: list) -> str:
    """
    Corrige os caracteres do texto de acordo com o formato esperado.

    Para cada posição:
    - Se o formato pede número ('N') mas veio uma letra, converte usando LETRA_PARA_NUMERO.
    - Se o formato pede letra ('L') mas veio um número, converte usando NUMERO_PARA_LETRA.
    - Caso contrário, mantém o caractere original.
    """
    if len(texto) != len(fmt):
        return texto
    resultado = []
    for char, tipo in zip(texto, fmt):
        if tipo == 'N' and not char.isdigit():
            resultado.append(LETRA_PARA_NUMERO.get(char, char))
        elif tipo == 'L' and not char.isalpha():
            resultado.append(NUMERO_PARA_LETRA.get(char, char))
        else:
            resultado.append(char)
    return ''.join(resultado)


def pos_processar(texto_bruto: str) -> str:
    """
    Aplica pós-processamento textual ao resultado bruto do OCR.

    Só corrige quando a correção resolve a ambiguidade de formato:
    - Se apenas o formato Antiga produz placa válida → aplica correção Antiga.
    - Se apenas o formato Mercosul produz placa válida → aplica correção Mercosul.
    - Se ambos ou nenhum produz placa válida → retorna o texto original sem alterar.
    """
    texto = texto_bruto.upper().replace(' ', '').replace('-', '').strip()

    if len(texto) != 7:
        return texto_bruto

    def e_valida_antiga(p):
        return len(p) == 7 and p[:3].isalpha() and p[3:].isdigit()

    def e_valida_mercosul(p):
        return (len(p) == 7 and p[:3].isalpha() and
                p[3].isdigit() and p[4].isalpha() and p[5:].isdigit())

    corrigido_antiga   = corrigir_por_formato(texto, FORMATO_ANTIGA)
    corrigido_mercosul = corrigir_por_formato(texto, FORMATO_MERCOSUL)

    valida_antiga   = e_valida_antiga(corrigido_antiga)
    valida_mercosul = e_valida_mercosul(corrigido_mercosul)

    if valida_antiga and not valida_mercosul:
        return corrigido_antiga
    if valida_mercosul and not valida_antiga:
        return corrigido_mercosul

    # Ambíguo ou nenhum válido — não arrisca
    return texto


# Teste rápido
print(pos_processar('FSA9I52'))  # ambíguo — válido nos dois formatos, não corrige
print(pos_processar('AUF3100'))  # ambíguo — válido nos dois formatos, não corrige
print(pos_processar('CON1454'))  # Antiga válida, Mercosul inválida → corrige para CON1454
print(pos_processar('RRR9100'))  # exemplo: posição 4 com '1' só é erro no Mercosul → RRR9I00

## 3. Avaliação com Dataset Real

In [ ]:
import pandas as pd
from fast_plate_ocr import LicensePlateRecognizer
import os

ocr_model = LicensePlateRecognizer('cct-s-v2-global-model')
dataset_path = './dataset/plates'
df = pd.read_csv(os.path.join(dataset_path, 'plates.csv'))

def normalizar(texto):
    return str(texto).upper().replace(' ', '').replace('-', '').strip()

def tipo_placa(texto):
    t = normalizar(texto)
    if len(t) == 7 and t[:3].isalpha() and t[3:].isdigit():
        return 'Antiga'
    elif len(t) == 7 and t[:3].isalpha() and t[3].isdigit() and t[4].isalpha() and t[5:].isdigit():
        return 'Mercosul'
    return 'Outro'

resultados = []
for _, row in df.iterrows():
    filename = str(row['filename'])
    gabarito = normalizar(row['plate'])
    img_path = os.path.join(dataset_path, filename)

    res = ocr_model.run(img_path)
    ocr_bruto     = normalizar(res[0].plate)
    ocr_corrigido = normalizar(pos_processar(ocr_bruto))

    resultados.append({
        'filename':          filename,
        'gabarito':          gabarito,
        'tipo':              tipo_placa(gabarito),
        'ocr_bruto':         ocr_bruto,
        'ocr_corrigido':     ocr_corrigido,
        'correto_bruto':     ocr_bruto     == gabarito,
        'correto_corrigido': ocr_corrigido == gabarito,
    })

resultado_df = pd.DataFrame(resultados)
print(f'Processadas {len(resultado_df)} imagens.')

## 4. Resultados

In [ ]:
total             = len(resultado_df)
acertos_bruto     = resultado_df['correto_bruto'].sum()
acertos_corrigido = resultado_df['correto_corrigido'].sum()

print('=' * 50)
print(f'  OCR bruto:          {acertos_bruto}/{total}  ({100*acertos_bruto/total:.1f}%)')
print(f'  OCR + heurísticas:  {acertos_corrigido}/{total}  ({100*acertos_corrigido/total:.1f}%)')
print(f'  Ganho:              +{acertos_corrigido - acertos_bruto} placas')
print('=' * 50)

for tipo in ['Antiga', 'Mercosul', 'Outro']:
    sub = resultado_df[resultado_df['tipo'] == tipo]
    if len(sub) == 0:
        continue
    b = sub['correto_bruto'].sum()
    c = sub['correto_corrigido'].sum()
    n = len(sub)
    print(f'  {tipo:10s}: {b}/{n} → {c}/{n}  (ganho: +{c-b})')

## 5. Análise de Erros e Regressões

In [ ]:
corrigidas = resultado_df[
    ~resultado_df['correto_bruto'] & resultado_df['correto_corrigido']
][['filename', 'gabarito', 'tipo', 'ocr_bruto', 'ocr_corrigido']]

regredidas = resultado_df[
    resultado_df['correto_bruto'] & ~resultado_df['correto_corrigido']
][['filename', 'gabarito', 'tipo', 'ocr_bruto', 'ocr_corrigido']]

print(f'Placas corrigidas com sucesso: {len(corrigidas)}')
display(corrigidas)

print(f'Placas que regrediram: {len(regredidas)}')
display(regredidas)

## 6. Visualização das Placas Corrigidas

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

n = min(len(corrigidas), 8)
if n == 0:
    print('Nenhuma placa foi corrigida.')
else:
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3))
    if n == 1:
        axes = [axes]
    for ax, (_, row) in zip(axes, corrigidas.head(n).iterrows()):
        img = mpimg.imread(os.path.join(dataset_path, row['filename']))
        ax.imshow(img)
        ax.set_title(
            f"Gabarito: {row['gabarito']}\nOCR: {row['ocr_bruto']}\nCorrigido: {row['ocr_corrigido']}",
            fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.show()